In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split the pdf in small chunks 
def load_and_docs_creation(pdf_path:'str'):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=100)
    docs = splitter.split_documents(documents)
    return docs



In [8]:
document_path = 'documents/budget_speech.pdf'
docs = load_and_docs_creation(document_path)

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()

# 1. Init Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "langchainvector"
index = pc.Index(index_name)

# 2. Setup embeddings (MiniLM → 384 dimension)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
dim = len(embeddings.embed_query("hello world"))
print("Embedding dimension:", dim)


Embedding dimension: 384


In [ ]:
# Delete old index if wrong dimension (optional cleanup)
if index_name in [i.name for i in pc.list_indexes()]:
    pc.delete_index(index_name)

# Create new index with correct dimension
pc.create_index(
    name=index_name,
    dimension=dim,   # must match embeddings it changes as per models
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
print("Index created:", index_name)

Index created: langchainvector


In [ ]:
vectorstore = PineconeVectorStore(
    index_name='langchainvector',
    embedding=embeddings,
    namespace="default"
)

# Upload the vectors to db
vectorstore.add_documents(documents)
print(" Docs uploaded")


 Docs uploaded


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

#model initialization for RAG 
model_id = "Qwen/Qwen2.5-1.5B-Instruct"   # small

# Load tokenizer
tok = AutoTokenizer.from_pretrained(model_id)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Load model (FP16 on GPU if available)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",        # auto place on GPU if available
    torch_dtype=torch.float16 # use FP16 to save VRAM
)

# Build pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    return_full_text=False    
)


Device set to use cpu


In [ ]:
def rag_answer(query, index, embeddings, tok, pipe, k=3, namespace="default",show_prompt=False):
    # 1. Embed query
    query_vec = embeddings.embed_query(query)

    # 2. Query Pinecone (this will give the matching document from the vector DB) many ways to query
    res = index.query(
        vector=query_vec,
        top_k=k,
        include_metadata=True,
        namespace=namespace
    )
    context = "\n\n".join(m["metadata"]["text"] for m in res["matches"]) 

    # 3. Build chat messages (system + user roles)
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"}
    ]

    # 4. Convert messages → prompt string using chat template
    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True   # ensures assistant starts response
    )
    if show_prompt:
        print(prompt)

    # 5. Send prompt to LLM
    answer = pipe(prompt)[0]["generated_text"].strip()
    return answer


In [96]:
query = "What is the development plan for next year give me a summary? "
answer = rag_answer(query, index, embeddings, tok, pipe)
print("Answer:", answer)


Answer: The budget proposes several initiatives focused on inclusive development, reaching the last mile, infrastructure and investment, unleashing potential, green growth, youth power, and financial sector. Key areas include digital public infrastructure for agriculture, targeting disadvantaged groups, focusing on rural development, and investing in renewable energy. The government aims to create green jobs and reduce carbon emissions while providing targeted support to various socio-economic segments including women, farmers, OBCs, SCs, STs, and economically weaker sections.


In [95]:
query = "How the government act on tobacco/cigrette sale give me a summary?"
answer = rag_answer(query, index, embeddings, tok, pipe,show_prompt=True)
print("Answer:", answer)


<|im_start|>system
You are a helpful assistant. Use only the provided context.<|im_end|>
<|im_start|>user
Context:
54 
 
 
 
D. CHANGES IN CENTRAL EXCISE 
D.1. NCCD Duty rate  on Cigarettes [with effect from 02.02.2023]  
 
Description of goods 
Rate of excise duty 
From 
(` per 1000 
sticks) 
To 
(` per 1000 
sticks) 
Other than filter cigarettes, of length not 
exceeding 65 mm 
200 230 
Other than filter cigarettes, of length exceeding 
65 mm but not exceeding 70 mm 
250 290 
Filter cigarettes of length not exceeding 65 mm 440 510 
Filter cigarettes of length exceeding 65 mm but 
not exceeding 70 mm 
440 510 
Filter cigarettes of length exceeding 70 mm but 
not exceeding 75 mm 
545 630 
Other cigarettes 735 850 
Cigarettes of tobacco substitutes 600 690 
 
 
D.2. Other changes in Central Excise [with effect from 02.02.2023] 
In order to promote green fuel, central excise duty exemption is being 
provided to blended Compressed Natural Gas from so much of the amount 
as is equal to the